# Global Wheat Detection — Data Exploration, Cleansing, Formatting & Augmentation

**Competition:** https://www.kaggle.com/competitions/global-wheat-detection/overview

**Internship timeline**
- Week 1 (this notebook): data exploration, cleansing, formatting, train/val split, and augmentation pipeline
- Week 3 & 4 (next notebook): fine-tuning and optimization, using the split + augmentation pipeline built here

This notebook covers:
1. Necessary libraries and data paths
2. Load and show data
3. Images with/without bounding boxes + example of each + total box count
4. Applying bounding boxes to sample images
5. Number of images by bounding-box count
6. Images and boxes per source
7. Bounding box area distribution + outlier detection (small / negative / large), calibrated visually
8. Aspect ratio distribution
9. Extracting and separating bounding box attributes (incl. duplicate/interchangeable box detection via IoU, and resolving which duplicate copy to drop)
10. Train / validation split (source-stratified, at image level)
11. Data augmentation pipeline (Albumentations) with visual sanity checks
12. Conversion to YOLO-format dataset (split-aware; outliers and duplicate boxes excluded)
13. YOLO label sanity check


## 1. Necessary Libraries and Data Paths

In [ ]:
import os
print(os.listdir("/kaggle/input/competitions"))

In [ ]:
import os
import ast
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image

import albumentations as A
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_NAME = "wheat"
CLASS_ID = 0

# ----- Data paths -----
# Adjust DATA_DIR to wherever the competition data was downloaded/extracted.
# Expected structure:
#   DATA_DIR/train.csv
#   DATA_DIR/train/*.jpg
#   DATA_DIR/test/*.jpg
DATA_DIR = Path("/kaggle/input/competitions/global-wheat-detection")          # e.g. Path("/kaggle/input/global-wheat-detection")
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMG_DIR = DATA_DIR / "train"
TEST_IMG_DIR = DATA_DIR / "test"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
YOLO_DIR = OUTPUT_DIR / "yolo_dataset"

assert TRAIN_CSV.exists(), f"train.csv not found at {TRAIN_CSV.resolve()} - update DATA_DIR"
print("Using data directory:", DATA_DIR.resolve())

## 2. Load and Show Data

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
print("train.csv shape:", df_raw.shape)
display(df_raw.head())
df_raw.info()
display(df_raw.describe(include="all"))

In [ ]:
all_train_images = sorted(p.stem for p in TRAIN_IMG_DIR.glob("*.jpg"))
print(f"Total images in train folder      : {len(all_train_images)}")
print(f"Total rows (boxes) in train.csv   : {len(df_raw)}")
print(f"Unique image_ids in train.csv     : {df_raw['image_id'].nunique()}")
print(f"Unique sources                    : {df_raw['source'].unique().tolist()}")

In [ ]:
# Peek at a handful of raw images (no boxes yet)
sample_ids = random.sample(all_train_images, 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_id in zip(axes.ravel(), sample_ids):
    img = Image.open(TRAIN_IMG_DIR / f"{img_id}.jpg")
    ax.imshow(img)
    ax.set_title(img_id, fontsize=9)
    ax.axis("off")
plt.suptitle("Sample raw training images")
plt.tight_layout()
plt.show()

## 3. Images With / Without Bounding Boxes

In [ ]:
def parse_bbox(bbox_str):
    """Parse the string-encoded bbox '[xmin, ymin, w, h]' into floats."""
    return ast.literal_eval(bbox_str)

df = df_raw.copy()
bbox_arr = np.array(df["bbox"].apply(parse_bbox).tolist())
df["x_min"] = bbox_arr[:, 0]
df["y_min"] = bbox_arr[:, 1]
df["box_width"] = bbox_arr[:, 2]
df["box_height"] = bbox_arr[:, 3]

images_with_boxes = set(df["image_id"].unique())
images_without_boxes = sorted(set(all_train_images) - images_with_boxes)

print(f"Images WITH at least one bounding box : {len(images_with_boxes)}")
print(f"Images WITHOUT any bounding box        : {len(images_without_boxes)}")
print(f"Total images                           : {len(all_train_images)}")
print(f"Total bounding boxes                   : {len(df)}")

In [ ]:
example_with = df["image_id"].iloc[0]
example_without = images_without_boxes[0] if images_without_boxes else None

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

img = np.array(Image.open(TRAIN_IMG_DIR / f"{example_with}.jpg"))
axes[0].imshow(img)
for _, row in df[df["image_id"] == example_with].iterrows():
    rect = patches.Rectangle((row.x_min, row.y_min), row.box_width, row.box_height,
                              linewidth=2, edgecolor="red", facecolor="none")
    axes[0].add_patch(rect)
axes[0].set_title(f"WITH boxes: {example_with}")
axes[0].axis("off")

if example_without:
    img2 = np.array(Image.open(TRAIN_IMG_DIR / f"{example_without}.jpg"))
    axes[1].imshow(img2)
    axes[1].set_title(f"WITHOUT boxes: {example_without}")
else:
    axes[1].text(0.5, 0.5, "No image without boxes found", ha="center")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 4. Applying Bounding Boxes to Sample Images

In [ ]:
def draw_boxes(image_id, boxes_df, ax=None, color="red", linewidth=2):
    img = np.array(Image.open(TRAIN_IMG_DIR / f"{image_id}.jpg"))
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img)
    boxes = boxes_df[boxes_df["image_id"] == image_id]
    for _, row in boxes.iterrows():
        rect = patches.Rectangle((row.x_min, row.y_min), row.box_width, row.box_height,
                                  linewidth=linewidth, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
    ax.set_title(f"{image_id}  ({len(boxes)} boxes)", fontsize=10)
    ax.axis("off")
    return ax

sample_with_boxes = random.sample(list(images_with_boxes), 6)
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, img_id in zip(axes.ravel(), sample_with_boxes):
    draw_boxes(img_id, df, ax=ax)
plt.suptitle("Sample images with bounding boxes")
plt.tight_layout()
plt.show()

## 5. Number of Images by Bounding-Box Count

In [ ]:
box_counts_per_image = df.groupby("image_id").size()
# include images that have zero boxes
box_counts_full = box_counts_per_image.reindex(all_train_images, fill_value=0)

plt.figure(figsize=(12, 6))
sns.histplot(box_counts_full, bins=range(0, int(box_counts_full.max()) + 2), color="steelblue")
plt.xlabel("Number of bounding boxes per image")
plt.ylabel("Number of images")
plt.title("Distribution of bounding-box counts per image")
plt.show()

print(box_counts_full.describe())
print(f"Images with 0 boxes: {(box_counts_full == 0).sum()}")

## 6. Images and Boxes per Source

In [ ]:
boxes_per_source = df["source"].value_counts()
images_per_source = df.groupby("source")["image_id"].nunique().reindex(boxes_per_source.index)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=images_per_source.index, y=images_per_source.values,
            hue=images_per_source.index, palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("Number of images per source")
axes[0].set_ylabel("Images")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(x=boxes_per_source.index, y=boxes_per_source.values,
            hue=boxes_per_source.index, palette="magma", legend=False, ax=axes[1])
axes[1].set_title("Number of bounding boxes per source")
axes[1].set_ylabel("Bounding boxes")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

display(pd.DataFrame({"images": images_per_source, "boxes": boxes_per_source}))

In [ ]:
# A few example images (with boxes drawn) from each source
sources = boxes_per_source.index.tolist()
n_examples = 3

fig, axes = plt.subplots(len(sources), n_examples, figsize=(5 * n_examples, 5 * len(sources)))
if len(sources) == 1:
    axes = axes[None, :]

for row_i, source in enumerate(sources):
    source_ids = df.loc[df["source"] == source, "image_id"].unique()
    sample_ids = random.sample(list(source_ids), min(n_examples, len(source_ids)))

    for col_i in range(n_examples):
        ax = axes[row_i, col_i]
        if col_i >= len(sample_ids):
            ax.axis("off")
            continue
        img_id = sample_ids[col_i]
        draw_boxes(img_id, df, ax=ax, color="lime")
        ax.set_title(f"{source} | {img_id}", fontsize=9)

plt.suptitle("Sample images with bounding boxes, per source", fontsize=16, y=1.001)
plt.tight_layout()
plt.show()

## 7. Bounding Box Area Distribution & Outliers

Box area is compared to the size of its **own** image (`area_ratio`), not a fixed
pixel count — a "big" box on a small image and a "big" box on a large image mean
very different things. Thresholds below were picked by visually calibrating
against real examples (see 7.2), not guessed upfront.

### 7.1 Compute Box Area Relative to Image Size

In [ ]:
df["area"] = df["box_width"] * df["box_height"]
df["image_area"] = df["width"] * df["height"]
df["area_ratio"] = df["area"] / df["image_area"]

NEG_DIM_MASK = (df["box_width"] <= 0) | (df["box_height"] <= 0)
print(f"Negative/zero-dimension boxes: {NEG_DIM_MASK.sum()}")

### 7.2 Area-Ratio Bucket Calibration

Before committing to small/large thresholds, look at real examples across the
full range of `area_ratio` to see where "normal" actually stops.

In [ ]:
df_valid = df[~NEG_DIM_MASK].copy()
df_valid["area_pct"] = df_valid["area_ratio"] * 100

bucket_edges = [0, 1, 10, 20, 30, 40, 50, 100]
bucket_labels = ["<1%", "1-10%", "10-20%", "20-30%", "30-40%", "40-50%", "50%+"]
df_valid["area_bucket"] = pd.cut(df_valid["area_pct"], bins=bucket_edges,
                                  labels=bucket_labels, right=False)

bucket_counts = df_valid["area_bucket"].value_counts().reindex(bucket_labels)
bucket_pct = (bucket_counts / len(df_valid) * 100).round(3)
display(pd.DataFrame({"count": bucket_counts, "% of all boxes": bucket_pct}))

plt.figure(figsize=(10, 5))
sns.barplot(x=bucket_counts.index, y=bucket_counts.values,
            hue=bucket_counts.index, palette="crest", legend=False)
plt.yscale("log")
plt.ylabel("Number of boxes (log scale)")
plt.xlabel("Box area as % of image area")
plt.title("Box count per area-ratio bucket")
plt.show()

In [ ]:
def show_bucket_examples(df_valid, bucket_labels, n_per_bucket=3, seed=RANDOM_SEED):
    n_rows = len(bucket_labels)
    fig, axes = plt.subplots(n_rows, n_per_bucket,
                              figsize=(5 * n_per_bucket, 5 * n_rows))
    if n_rows == 1:
        axes = axes[None, :]

    for row_i, bucket in enumerate(bucket_labels):
        bucket_df = df_valid[df_valid["area_bucket"] == bucket]
        sampled_rows = bucket_df.sample(min(n_per_bucket, len(bucket_df)), random_state=seed) \
            if len(bucket_df) > 0 else bucket_df

        for col_i in range(n_per_bucket):
            ax = axes[row_i, col_i]
            ax.axis("off")
            if col_i >= len(sampled_rows):
                if col_i == 0 and len(sampled_rows) == 0:
                    ax.text(0.5, 0.5, f"No boxes in {bucket}", ha="center", va="center")
                continue

            row = sampled_rows.iloc[col_i]
            img = np.array(Image.open(TRAIN_IMG_DIR / f"{row.image_id}.jpg"))
            ax.imshow(img)
            rect = patches.Rectangle((row.x_min, row.y_min), row.box_width, row.box_height,
                                      linewidth=3, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.set_title(f"{bucket} | {row.image_id}\n{row.area_pct:.2f}% of image", fontsize=9)

    plt.suptitle("Example boxes across area-ratio buckets", fontsize=16, y=1.002)
    plt.tight_layout()
    plt.show()

show_bucket_examples(df_valid, bucket_labels, n_per_bucket=3)

### 7.3 Final Outlier Thresholds

In [ ]:
SMALL_AREA_RATIO_THRESH = 0.0005   # box covers < 0.05% of its image
LARGE_AREA_RATIO_THRESH = 0.15     # box covers > 15% of its image

SMALL_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] < SMALL_AREA_RATIO_THRESH)
LARGE_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] > LARGE_AREA_RATIO_THRESH)
NORMAL_MASK = ~(NEG_DIM_MASK | SMALL_MASK | LARGE_MASK)

print(f"Negative/zero-dimension boxes : {NEG_DIM_MASK.sum()}")
print(f"Small outlier boxes (< {SMALL_AREA_RATIO_THRESH:.3%} of image) : {SMALL_MASK.sum()}")
print(f"Large outlier boxes (> {LARGE_AREA_RATIO_THRESH:.1%} of image) : {LARGE_MASK.sum()}")
print(f"Normal boxes                                        : {NORMAL_MASK.sum()}")

plt.figure(figsize=(12, 6))
plt.hist(df.loc[NORMAL_MASK, "area_ratio"] * 100, bins=80, color="steelblue", alpha=0.8, label="normal")
plt.hist(df.loc[SMALL_MASK, "area_ratio"] * 100, bins=20, color="orange", alpha=0.9, label="small outlier")
plt.hist(df.loc[LARGE_MASK, "area_ratio"] * 100, bins=20, color="red", alpha=0.9, label="large outlier")
plt.axvline(SMALL_AREA_RATIO_THRESH * 100, color="orange", linestyle="--", linewidth=1)
plt.axvline(LARGE_AREA_RATIO_THRESH * 100, color="red", linestyle="--", linewidth=1)
plt.yscale("log")
plt.xlabel("Bounding box area as % of its image's area")
plt.ylabel("Number of boxes (log scale)")
plt.title("Bounding box area (relative to image size) with outliers highlighted")
plt.legend()
plt.show()

### 7.4 Visualize Final Small vs Large Outliers

In [ ]:
def show_outlier_columns(small_mask, large_mask, n=5, colors=("blue", "red")):
    def sample_rows(mask, n):
        sub = df.loc[mask]
        ids = sub["image_id"].unique()
        sample_ids = random.sample(list(ids), min(n, len(ids)))
        return [sub[sub["image_id"] == img_id].iloc[0] for img_id in sample_ids]

    small_examples = sample_rows(small_mask, n)
    large_examples = sample_rows(large_mask, n)

    fig, axes = plt.subplots(n, 2, figsize=(12, 6 * n))
    axes = np.atleast_2d(axes)

    for col_i, (examples, mask, color, col_title) in enumerate([
        (small_examples, small_mask, colors[0], "Small-area outlier"),
        (large_examples, large_mask, colors[1], "Large-area outlier"),
    ]):
        for row_i in range(n):
            ax = axes[row_i, col_i]
            if row_i >= len(examples):
                ax.axis("off")
                continue

            ex_row = examples[row_i]
            img_id = ex_row.image_id
            img = np.array(Image.open(TRAIN_IMG_DIR / f"{img_id}.jpg"))
            ax.imshow(img)

            rows = df[(df["image_id"] == img_id) & mask]
            for _, row in rows.iterrows():
                rect = patches.Rectangle(
                    (row.x_min, row.y_min),
                    max(row.box_width, 1), max(row.box_height, 1),
                    linewidth=2.5, edgecolor=color, facecolor="none")
                ax.add_patch(rect)

            ax.set_title(f"{col_title} | {img_id}\n{ex_row.area_ratio:.2%} of image", fontsize=9)
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_outlier_columns(SMALL_MASK, LARGE_MASK)

## 8. Aspect Ratio Distribution

In [ ]:
valid = df[~NEG_DIM_MASK].copy()
valid["aspect_ratio"] = valid["box_width"] / valid["box_height"]

plt.figure(figsize=(12, 6))
sns.histplot(valid["aspect_ratio"], bins=100, color="teal")
plt.xlim(0, 5)
plt.axvline(1.0, color="black", linestyle="--", label="square (1:1)")
plt.xlabel("Aspect ratio (width / height)")
plt.ylabel("Number of boxes")
plt.title("Bounding box aspect ratio distribution")
plt.legend()
plt.show()

print(valid["aspect_ratio"].describe())

## 9. Extracting and Separating Bounding Box Attributes

Consolidate every box attribute derived above (coordinates, area, aspect ratio,
outlier flags) into a single clean, well-typed dataframe. It is saved once,
at the end of Section 9.1, after the duplicate-box decision below is folded in.

In [ ]:
clean_df = df.copy()
clean_df["x_max"] = clean_df["x_min"] + clean_df["box_width"]
clean_df["y_max"] = clean_df["y_min"] + clean_df["box_height"]
clean_df["area"] = clean_df["box_width"] * clean_df["box_height"]
clean_df["aspect_ratio"] = clean_df["box_width"] / clean_df["box_height"].replace(0, np.nan)

clean_df["is_negative_dim"] = NEG_DIM_MASK
clean_df["is_small_outlier"] = SMALL_MASK
clean_df["is_large_outlier"] = LARGE_MASK
clean_df["is_outlier"] = NEG_DIM_MASK | SMALL_MASK | LARGE_MASK

attribute_cols = [
    "image_id", "width", "height", "source",
    "x_min", "y_min", "box_width", "box_height", "x_max", "y_max",
    "area", "aspect_ratio",
    "is_negative_dim", "is_small_outlier", "is_large_outlier", "is_outlier",
]
clean_df = clean_df[attribute_cols]

display(clean_df.head())
print(f"Outlier boxes flagged: {clean_df['is_outlier'].sum()} / {len(clean_df)}")

### 9.1 Duplicate / Interchangeable Box Detection (IoU)

Two boxes on the same image with very high IoU are essentially duplicate
annotations of the same wheat head. Detected here via pairwise IoU per image.
For each duplicate pair, the **first** box is kept and the **second** is marked
for exclusion — keeping both would let the model be "rewarded twice" for the
same wheat head during training.

In [ ]:
IOU_DUPLICATE_THRESH = 0.65  # boxes overlapping more than this are treated as interchangeable/duplicate

def iou_matrix(x_min, y_min, x_max, y_max):
    areas = (x_max - x_min) * (y_max - y_min)
    xx1 = np.maximum(x_min[:, None], x_min[None, :])
    yy1 = np.maximum(y_min[:, None], y_min[None, :])
    xx2 = np.minimum(x_max[:, None], x_max[None, :])
    yy2 = np.minimum(y_max[:, None], y_max[None, :])
    inter_w = np.clip(xx2 - xx1, 0, None)
    inter_h = np.clip(yy2 - yy1, 0, None)
    inter = inter_w * inter_h
    union = areas[:, None] + areas[None, :] - inter
    return np.where(union > 0, inter / union, 0)

dup_records = []
for image_id, group in clean_df.groupby("image_id"):
    if len(group) < 2:
        continue
    x_min, y_min = group["x_min"].to_numpy(), group["y_min"].to_numpy()
    x_max, y_max = group["x_max"].to_numpy(), group["y_max"].to_numpy()
    ious = iou_matrix(x_min, y_min, x_max, y_max)

    idx_i, idx_j = np.triu_indices(len(group), k=1)
    pair_ious = ious[idx_i, idx_j]
    dup_mask = pair_ious > IOU_DUPLICATE_THRESH

    for i, j, iou_val in zip(idx_i[dup_mask], idx_j[dup_mask], pair_ious[dup_mask]):
        dup_records.append({
            "image_id": image_id,
            "source": group["source"].iloc[0],
            "box_i": group.index[i],
            "box_j": group.index[j],
            "iou": iou_val,
        })

dup_df = pd.DataFrame(dup_records)
print(f"Total interchangeable (duplicate) box pairs at IoU > {IOU_DUPLICATE_THRESH}: {len(dup_df)}")

if len(dup_df) > 0:
    print(f"Images affected: {dup_df['image_id'].nunique()}")

    dup_per_source = dup_df.groupby("source").size().rename("duplicate_pairs")
    dup_per_image = dup_df.groupby("image_id").size().sort_values(ascending=False).rename("duplicate_pairs")

    display(dup_per_source.to_frame())
    display(dup_per_image.head(10).to_frame())

    plt.figure(figsize=(10, 5))
    sns.barplot(x=dup_per_source.index, y=dup_per_source.values,
                hue=dup_per_source.index, palette="rocket", legend=False)
    plt.ylabel("Duplicate box pairs")
    plt.title(f"Interchangeable box pairs per source (IoU > {IOU_DUPLICATE_THRESH})")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("No interchangeable boxes found at this threshold.")

In [ ]:
# --- 9.0.5 IoU Calibration: what do pairs at different overlap levels actually look like? ---

def all_pairwise_ious(boxes_df):
    """Compute IoU for every same-image box pair, return as a flat dataframe."""
    records = []
    for image_id, group in boxes_df.groupby("image_id"):
        if len(group) < 2:
            continue
        x_min, y_min = group["x_min"].to_numpy(), group["y_min"].to_numpy()
        x_max, y_max = group["x_max"].to_numpy(), group["y_max"].to_numpy()
        ious = iou_matrix(x_min, y_min, x_max, y_max)
        idx_i, idx_j = np.triu_indices(len(group), k=1)
        for i, j, v in zip(idx_i, idx_j, ious[idx_i, idx_j]):
            if v > 0:  # skip completely non-overlapping pairs
                records.append({
                    "image_id": image_id,
                    "box_i": group.index[i], "box_j": group.index[j], "iou": v,
                })
    return pd.DataFrame(records)

# clean_df must already have x_max/y_max (from Section 9) before this runs
all_pairs_df = all_pairwise_ious(clean_df)
print(f"Total overlapping box pairs found: {len(all_pairs_df)}")

iou_bands = [(0.3, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]
n_per_band = 3

fig, axes = plt.subplots(len(iou_bands), n_per_band, figsize=(5 * n_per_band, 5 * len(iou_bands)))

for row_i, (lo, hi) in enumerate(iou_bands):
    band_df = all_pairs_df[(all_pairs_df["iou"] >= lo) & (all_pairs_df["iou"] < hi)]
    sampled = band_df.sample(min(n_per_band, len(band_df)), random_state=RANDOM_SEED) if len(band_df) else band_df

    for col_i in range(n_per_band):
        ax = axes[row_i, col_i]
        ax.axis("off")
        if col_i >= len(sampled):
            if col_i == 0 and len(sampled) == 0:
                ax.text(0.5, 0.5, f"No pairs in [{lo:.1f}, {hi:.1f})", ha="center", va="center")
            continue

        rec = sampled.iloc[col_i]
        img = np.array(Image.open(TRAIN_IMG_DIR / f"{rec.image_id}.jpg"))
        ax.imshow(img)
        for box_idx, color in [(rec.box_i, "red"), (rec.box_j, "cyan")]:
            b = clean_df.loc[box_idx]
            rect = patches.Rectangle((b.x_min, b.y_min), b.box_width, b.box_height,
                                      linewidth=2.5, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
        ax.set_title(f"IoU=[{lo:.1f},{hi:.1f}) | {rec.image_id}\nactual={rec.iou:.2f}", fontsize=9)

plt.suptitle("Box-pair overlap calibration: same head (duplicate) vs. two real, touching heads?", fontsize=15, y=1.001)
plt.tight_layout()
plt.show()

# Quick distribution check to see how many pairs fall in the ambiguous zone
print(all_pairs_df["iou"].describe())
print("\nPairs per band:")
for lo, hi in iou_bands:
    n = ((all_pairs_df["iou"] >= lo) & (all_pairs_df["iou"] < hi)).sum()
    print(f"  [{lo:.1f}, {hi:.1f}): {n}")

In [ ]:

clean_df["use_for_training"] = ~(clean_df["is_outlier"])

clean_csv_path = OUTPUT_DIR / "train_boxes_clean.csv"
clean_df.to_csv(clean_csv_path, index=False)

print(f"Saved cleaned, attribute-separated data to: {clean_csv_path.resolve()}")
print(f"Outlier boxes flagged             : {clean_df['is_outlier'].sum()} / {len(clean_df)}")
print(f"Boxes usable for training         : {clean_df['use_for_training'].sum()} / {len(clean_df)}")

## 10. Train / Validation Split (Source-Stratified)

The split is done **per image** (not per box) and **stratified by source**, since
box density and visual characteristics vary noticeably by source (Section 6).
Images with zero boxes are grouped into their own pseudo-source ("no_box") so
they are distributed proportionally across train/val too, rather than landing
disproportionately in one split.

In [ ]:
image_source = (
    df.groupby("image_id")["source"]
    .first()
    .reindex(all_train_images)
    .fillna("no_box")
)

image_level_df = (
    image_source.rename("source")
    .to_frame()
    .reset_index()
    .rename(columns={"index": "image_id"})
)

RANDOM_SEED = 42

# First split: 80% train, 20% temporary
train_ids, temp_ids = train_test_split(
    image_level_df["image_id"],
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=image_level_df["source"],
)

# Second split: divide temporary set into 10% validation and 10% test
temp_sources = image_level_df.set_index("image_id").loc[temp_ids, "source"]

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_sources,
)

train_ids = set(train_ids)
val_ids = set(val_ids)
test_ids = set(test_ids)

image_level_df["split"] = image_level_df["image_id"].apply(
    lambda image_id: (
        "train" if image_id in train_ids
        else "val" if image_id in val_ids
        else "test"
    )
)

print(image_level_df["split"].value_counts())
display(pd.crosstab(image_level_df["source"], image_level_df["split"]))

split_csv_path = OUTPUT_DIR / "image_split.csv"
image_level_df.to_csv(split_csv_path, index=False)

print(f"Saved split assignment to: {split_csv_path.resolve()}")

split_lookup = image_level_df.set_index("image_id")["split"]

## 12. Convert to YOLO-Format Dataset

Builds a split-aware `images/{train,val}` + `labels/{train,val}` YOLO dataset
(single class: `wheat`):
- One `.txt` label file per image (empty file for images with no usable boxes)
- Each line: `class x_center y_center width height` (all normalized 0-1)
- Boxes excluded from labels: outliers (Section 7) **and** the dropped copy of each duplicate pair (Section 9.1) — i.e. only `use_for_training == True` boxes are written
- Images are written into `train/` or `val/` per the Section 10 split assignment

The augmentation pipeline from Section 11 is **not** applied here — it stays as
a pipeline object the Week 3/4 training script imports and runs on-the-fly,
train split only.

In [ ]:
from sklearn.model_selection import train_test_split

# --- Build an 80/10/10 split, stratified by source ---
image_source_df = clean_df[["image_id", "source"]].drop_duplicates()

train_ids, temp_ids = train_test_split(
    image_source_df["image_id"], test_size=0.20,
    stratify=image_source_df["source"], random_state=RANDOM_SEED,
)
temp_source = image_source_df.set_index("image_id").loc[temp_ids, "source"]
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, stratify=temp_source, random_state=RANDOM_SEED,
)  # 0.5 of the 20% held out -> 10% val, 10% test

# images with no boxes have no known source - split them the same way, unstratified
no_box_ids = sorted(set(all_train_images) - set(image_source_df["image_id"]))
nb_train, nb_temp = train_test_split(no_box_ids, test_size=0.20, random_state=RANDOM_SEED)
nb_val, nb_test = train_test_split(nb_temp, test_size=0.50, random_state=RANDOM_SEED)

split_lookup = {}
for img_id in list(train_ids) + nb_train:
    split_lookup[img_id] = "train"
for img_id in list(val_ids) + nb_val:
    split_lookup[img_id] = "val"
for img_id in list(test_ids) + nb_test:
    split_lookup[img_id] = "test"

print(pd.Series(split_lookup).value_counts())

In [ ]:
YOLO_IMG_DIR = {s: YOLO_DIR / "images" / s for s in ["train", "val", "test"]}
YOLO_LBL_DIR = {s: YOLO_DIR / "labels" / s for s in ["train", "val", "test"]}
for d in list(YOLO_IMG_DIR.values()) + list(YOLO_LBL_DIR.values()):
    d.mkdir(parents=True, exist_ok=True)

def to_yolo_line(row, img_w, img_h):
    x_center = (row.x_min + row.box_width / 2) / img_w
    y_center = (row.y_min + row.box_height / 2) / img_h
    w = row.box_width / img_w
    h = row.box_height / img_h
    x_center, y_center, w, h = (float(np.clip(v, 0, 1)) for v in (x_center, y_center, w, h))
    return f"{CLASS_ID} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

boxes_for_yolo = clean_df[clean_df["use_for_training"]]

for img_id in all_train_images:
    split = split_lookup.get(img_id, "train")  # safe now: "train"/"val"/"test" all exist as keys
    rows = boxes_for_yolo[boxes_for_yolo["image_id"] == img_id]

    img_w, img_h = 1024, 1024
    if len(rows) > 0:
        img_w = int(rows.iloc[0]["width"])
        img_h = int(rows.iloc[0]["height"])

    lines = [to_yolo_line(r, img_w, img_h) for r in rows.itertuples()]
    (YOLO_LBL_DIR[split] / f"{img_id}.txt").write_text("\n".join(lines))

    src_img = TRAIN_IMG_DIR / f"{img_id}.jpg"
    dst_img = YOLO_IMG_DIR[split] / f"{img_id}.jpg"
    if not dst_img.exists():
        try:
            os.link(src_img, dst_img)
        except OSError:
            shutil.copy(src_img, dst_img)

data_yaml = f"""path: {YOLO_DIR.resolve()}
train: images/train
val: images/val
test: images/test
names:
  0: {CLASS_NAME}
"""
(YOLO_DIR / "data.yaml").write_text(data_yaml)

for split in ["train", "val", "test"]:
    n_img = len(list(YOLO_IMG_DIR[split].glob("*.jpg")))
    n_lbl = len(list(YOLO_LBL_DIR[split].glob("*.txt")))
    print(f"{split:5s}: {n_img} images, {n_lbl} labels")

## 13. YOLO Label Sanity Check

In [ ]:
def verify_yolo_label(image_id):
    split = split_lookup.get(image_id, "train")
    img = np.array(Image.open(TRAIN_IMG_DIR / f"{image_id}.jpg"))
    h, w = img.shape[:2]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    # ---- Left: original COCO-style representation [x_min, y_min, w, h] ----
    coco_rows = boxes_for_yolo[boxes_for_yolo["image_id"] == image_id]
    axes[0].imshow(img)
    for _, row in coco_rows.iterrows():
        rect = patches.Rectangle((row.x_min, row.y_min), row.box_width, row.box_height,
                                  linewidth=2, edgecolor="red", facecolor="none")
        axes[0].add_patch(rect)
    axes[0].set_title(f"COCO-style (original)\n{image_id}  |  split={split}  |  {len(coco_rows)} boxes")
    axes[0].axis("off")

    # ---- Right: re-drawn from the written YOLO label file ----
    lines = (YOLO_LBL_DIR[split] / f"{image_id}.txt").read_text().splitlines()
    axes[1].imshow(img)
    for line in lines:
        if not line.strip():
            continue
        cls, xc, yc, bw, bh = map(float, line.split())
        x_min = (xc - bw / 2) * w
        y_min = (yc - bh / 2) * h
        rect = patches.Rectangle((x_min, y_min), bw * w, bh * h,
                                  linewidth=2, edgecolor="lime", facecolor="none")
        axes[1].add_patch(rect)
    axes[1].set_title(f"YOLO-format (reconstructed)\n{image_id}  |  {len(lines)} boxes")
    axes[1].axis("off")

    plt.suptitle("COCO vs YOLO representation — sanity check")
    plt.tight_layout()
    plt.show()

check_id = next(iter(boxes_for_yolo["image_id"].unique()))
verify_yolo_label(check_id)

In [ ]:
!pip install -U ultralytics

from ultralytics import YOLO

# Load the latest YOLO26 small model
model = YOLO("yolo26s.pt")

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="global_wheat",
    name="yolo26s_augmented_fast",
    
    # # 🎨 AUGMENTATION (Tuned for Wheat / Dense Small Objects)
    # hsv_h=0.015,        # Hue shift (handles different times of day/lighting)
    # hsv_s=0.7,          # Saturation shift
    # hsv_v=0.4,          # Value/brightness shift (handles shadows/overexposure)
    # degrees=10.0,       # Slight rotation (wheat is mostly upright, keep low)
    # translate=0.1,      # Translation fraction
    # scale=0.5,          # Scale augmentation (helps with varying distances/sizes)
    # shear=0.0,          # Keep shear at 0 for natural agricultural images
    # perspective=0.001,  # Minimal perspective distortion
    # flipud=0.0,         # ⚠️ Vertical flip OFF (wheat doesn't grow upside down!)
    # fliplr=0.5,         # Horizontal flip ON (safe and highly effective)
    # mosaic=1.0,         # Mosaic ON (critical for small object detection)
    # mixup=0.1,          # MixUp (helps model learn to separate overlapping heads)
    # copy_paste=0.1,     # Copy-paste augmentation
    # close_mosaic=10,    # 🧠 Pro trick: Disable mosaic in last 10 epochs for finer tuning

    # 📉 EARLY STOPPING & LOGGING (Saves hours of wasted compute)
    patience=10,        # Stop training if validation metrics don't improve for 10 epochs
    save_period=5,      # Save checkpoints every 5 epochs (reduces disk write overhead)
    plots=True,         # Generate training plots
    exist_ok=True,      # Prevents errors if you need to re-run the same cell
)

In [ ]:
from ultralytics import YOLO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ✅ CORRECT PATH FOUND BY SEARCH
model = YOLO("/kaggle/working/runs/detect/global_wheat/yolo26s_augmented_fast/weights/best.pt")

print("="*70)
print("🎯 COMPREHENSIVE PREDICTION METRICS")
print("="*70)

# Run predictions on validation set
results = model.predict(
    source="/kaggle/working/outputs/yolo_dataset/images/val", # <--- THE FIX
    imgsz=640,
    batch=16,
    save=False,
    verbose=False,
    conf=0.25 # Only count predictions with >25% confidence
)

# Collect prediction data
all_confidences = []
all_boxes = []
total_predictions = 0
images_with_predictions = 0
images_without_predictions = 0

for result in results:
    total_predictions += len(result.boxes) if result.boxes is not None else 0
    if result.boxes is not None and len(result.boxes) > 0:
        images_with_predictions += 1
        all_confidences.extend(result.boxes.conf.cpu().numpy().tolist())
        all_boxes.extend(result.boxes.xyxy.cpu().numpy().tolist())
    else:
        images_without_predictions += 1

print(f"\n📊 PREDICTION STATISTICS")
print(f"{'='*70}")
print(f"Total images processed: {len(results)}")
print(f"Images with predictions: {images_with_predictions}")
print(f"Images without predictions: {images_without_predictions}")
print(f"Total bounding boxes predicted: {total_predictions}")
print(f"Average boxes per image: {total_predictions/len(results):.2f}")

if all_confidences:
    print(f"\n📈 CONFIDENCE SCORE ANALYSIS")
    print(f"{'='*70}")
    print(f"Mean confidence: {np.mean(all_confidences):.4f}")
    print(f"Median confidence: {np.median(all_confidences):.4f}")
    print(f"Min confidence: {np.min(all_confidences):.4f}")
    print(f"Max confidence: {np.max(all_confidences):.4f}")
    
    high_conf = sum(1 for c in all_confidences if c >= 0.75)
    med_conf = sum(1 for c in all_confidences if 0.5 <= c < 0.75)
    low_conf = sum(1 for c in all_confidences if c < 0.5)
    
    print(f"\nConfidence Distribution:")
    print(f"  High (≥0.75): {high_conf} ({high_conf/len(all_confidences)*100:.1f}%)")
    print(f"  Medium (0.5-0.75): {med_conf} ({med_conf/len(all_confidences)*100:.1f}%)")
    print(f"  Low (<0.5): {low_conf} ({low_conf/len(all_confidences)*100:.1f}%)")

# Box size analysis
if all_boxes:
    print(f"\n📦 BOUNDING BOX SIZE ANALYSIS")
    print(f"{'='*70}")
    
    box_sizes = []
    box_aspect_ratios = []
    
    for box in all_boxes:
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        area = width * height
        aspect_ratio = width / (height + 1e-6)
        
        box_sizes.append(area)
        box_aspect_ratios.append(aspect_ratio)
    
    print(f"Mean box area: {np.mean(box_sizes):.2f} pixels²")
    print(f"Median box area: {np.median(box_sizes):.2f} pixels²")
    print(f"Min box area: {np.min(box_sizes):.2f} pixels²")
    print(f"Max box area: {np.max(box_sizes):.2f} pixels²")
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.hist(box_sizes, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    plt.xlabel('Box Area (pixels²)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Bounding Box Sizes')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.hist(box_aspect_ratios, bins=50, edgecolor='black', alpha=0.7, color='coral')
    plt.xlabel('Aspect Ratio (width/height)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Box Aspect Ratios')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Confidence vs Box size correlation
if all_confidences and all_boxes:
    print(f"\n🔍 CONFIDENCE vs BOX SIZE CORRELATION")
    print(f"{'='*70}")
    
    box_areas = [(b[2] - b[0]) * (b[3] - b[1]) for b in all_boxes]
    correlation = np.corrcoef(box_areas, all_confidences)[0, 1]
    print(f"Correlation between box size and confidence: {correlation:.4f}")
    
    if correlation > 0.3:
        print("⚠️  Model is more confident on LARGER objects (might miss tiny wheat heads)")
    elif correlation < -0.3:
        print("⚠️  Model is more confident on SMALLER objects")
    else:
        print("✅ Confidence is independent of object size (Excellent!)")
    
    plt.figure(figsize=(10, 6))
    plt.scatter(box_areas, all_confidences, alpha=0.5, s=20, color='purple')
    plt.xlabel('Box Area (pixels²)')
    plt.ylabel('Confidence Score')
    plt.title('Confidence vs Object Size')
    plt.grid(True, alpha=0.3)
    plt.show()

# Final Validation Metrics
print(f"\n📊 FINAL VALIDATION METRICS")
print(f"{'='*70}")
val_metrics = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", imgsz=640, verbose=False)

print(f"mAP@50:    {val_metrics.box.map50:.4f}")
print(f"mAP@75:    {val_metrics.box.map75:.4f}")
print(f"mAP@50-95: {val_metrics.box.map:.4f}")
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")
f1 = 2 * (val_metrics.box.mp * val_metrics.box.mr) / (val_metrics.box.mp + val_metrics.box.mr + 1e-16)
print(f"F1-Score:  {f1:.4f}")
print(f"{'='*70}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import random

print("="*70)
print("🖼️ VISUALIZING SAMPLE PREDICTIONS")
print("="*70)

# ✅ CORRECT PATH: images/val (not valid/images)
val_dir = Path("/kaggle/working/outputs/yolo_dataset/images/val")
print(f" Checking directory: {val_dir}")
print(f"✅ Directory exists: {val_dir.exists()}")

# Get images (Check for BOTH .jpg and .png)
val_images = list(val_dir.glob("*.jpg")) + list(val_dir.glob("*.png"))
print(f"🖼️ Found {len(val_images)} validation images in total.\n")

if len(val_images) == 0:
    print("❌ ERROR: No images found!")
    print("💡 Check if the path is correct")
else:
    # Pick up to 8 random images
    num_to_show = min(8, len(val_images))
    sample_images = random.sample(val_images, num_to_show)

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for idx, img_path in enumerate(sample_images):
        # Run prediction
        result = model.predict(str(img_path), imgsz=640, conf=0.25, verbose=False)[0]
        
        # Get image with boxes drawn
        plotted_img = result.plot()
        
        # Convert BGR (OpenCV) to RGB (Matplotlib)
        img_rgb = cv2.cvtColor(plotted_img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img_rgb)
        axes[idx].axis('off')
        
        # Count boxes and add to title
        num_boxes = len(result.boxes) if result.boxes is not None else 0
        short_name = img_path.name[:15] + "..." if len(img_path.name) > 15 else img_path.name
        axes[idx].set_title(f"{short_name}\nPredictions: {num_boxes}", fontsize=10, fontweight='bold')

    # Hide unused subplots
    for idx in range(num_to_show, 8):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()
    print("\n✅ Visualization complete!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("="*70)
print("📈 TRAINING HEALTH & NEXT STEPS")
print("="*70)

# ✅ CORRECT PATH FOUND BY SEARCH
results_csv = Path("/kaggle/working/runs/detect/global_wheat/yolo26s_augmented_fast/results.csv")

if results_csv.exists():
    df = pd.read_csv(results_csv)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot mAP
    axes[0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95', color='blue', linewidth=2)
    axes[0].set_title('Model Accuracy Over Time')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('mAP@50-95')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot Loss
    axes[1].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='orange')
    axes[1].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='red', linewidth=2)
    axes[1].set_title('Loss Over Time')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Actionable Advice
    print("\n💡 ANALYSIS & NEXT STEPS:")
    print("-" * 70)
    
    train_loss_end = df['train/box_loss'].iloc[-1]
    val_loss_end = df['val/box_loss'].iloc[-1]
    
    if val_loss_end > train_loss_end * 1.3:
        print("⚠️ OVERFITTING DETECTED: Validation loss is much higher than training loss.")
        print("   ➔ FIX: Add slight dropout, reduce epochs, or add very light augmentation.")
    else:
        print("✅ HEALTHY TRAINING: Train and Validation losses are well-aligned.")
        
    last_5_epochs_map = df['metrics/mAP50-95(B)'].iloc[-5:].mean()
    if df['metrics/mAP50-95(B)'].iloc[-1] > last_5_epochs_map + 0.005:
        print("📈 TREND: mAP is still rising at the end of training.")
        print("   ➔ FIX: You could likely get better results by training for more epochs (e.g., 100).")
    else:
        print("✅ CONVERGENCE: Model has fully converged. No need for more epochs.")

else:
    print("❌ Could not find results.csv. Please check the path.")

In [ ]:
model = YOLO("yolo26s.pt")

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="global_wheat",
    name="yolo26s_tuned_week4_boxfix",
    exist_ok=True,

    box=8.5,            # FIXED — was 0.09, now correctly ABOVE default (7.5) to push tighter boxes
    dfl=1.75,            # slightly softened from your 2.0, paired properly with box now
    cls=0.5,             # default, unchanged
    lr0=0.01,
    lrf=0.005,
    weight_decay=0.0005,

    patience=10,
    save_period=5,
    plots=True,
)

In [ ]:
from ultralytics import YOLO

# Load your tuned model
model = YOLO("/kaggle/working/runs/detect/global_wheat/yolo26s_tuned_week4_boxfix/weights/best.pt")

print(" Running Test-Time Augmentation (TTA) Ensemble...")
print("="*70)

# Standard validation
standard = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", 
                     imgsz=640, verbose=False)

# TTA validation (looks at flipped/scaled versions too)
tta = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", 
                imgsz=640, augment=True, verbose=False)

print("\n📊 COMPARISON: Standard vs TTA Ensemble")
print("="*70)
print(f"{'Metric':<15} {'Standard':<15} {'TTA':<15} {'Improvement':<15}")
print("-"*70)

std_mAP50 = standard.box.map50
std_mAP75 = standard.box.map75
std_mAP = standard.box.map
std_f1 = 2 * (standard.box.mp * standard.box.mr) / (standard.box.mp + standard.box.mr + 1e-16)

tta_mAP50 = tta.box.map50
tta_mAP75 = tta.box.map75
tta_mAP = tta.box.map
tta_f1 = 2 * (tta.box.mp * tta.box.mr) / (tta.box.mp + tta.box.mr + 1e-16)

print(f"{'mAP@50':<15} {std_mAP50:<15.4f} {tta_mAP50:<15.4f} {tta_mAP50-std_mAP50:<15.4f}")
print(f"{'mAP@75':<15} {std_mAP75:<15.4f} {tta_mAP75:<15.4f} {tta_mAP75-std_mAP75:<15.4f}")
print(f"{'mAP@50-95':<15} {std_mAP:<15.4f} {tta_mAP:<15.4f} {tta_mAP-std_mAP:<15.4f}")
print(f"{'F1-Score':<15} {std_f1:<15.4f} {tta_f1:<15.4f} {tta_f1-std_f1:<15.4f}")
print("="*70)

In [ ]:
from ultralytics import YOLO

# Load the model
model = YOLO("yolo26s.pt")

# Run the automated hyperparameter search
model.tune(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=10,         # Keep low (10) just to find the best settings quickly
    iterations=10,     # Tests 10 different combinations
    imgsz=480,         # Keep low (480) to make the search run much faster
    batch=32,          # High batch for speed
    optimizer="AdamW", # Good optimizer for tuning
    plots=False,       # Save time by not generating plots during search
    save=False,        # Don't save the 10 temporary models, just the results
    project="global_wheat",
    name="hyperparameter_search",
    exist_ok=True,
)

In [ ]:
import yaml
from pathlib import Path

print("="*70)
print("🏆 EXTRACTING BEST HYPERPARAMETERS")
print("="*70)

# Path to the best hyperparameters YAML file
yaml_path = "/kaggle/working/runs/detect/global_wheat/hyperparameter_search/best_hyperparameters.yaml"

if Path(yaml_path).exists():
    with open(yaml_path, 'r') as f:
        best_params = yaml.safe_load(f)
    
    print("✅ Best hyperparameters loaded!")
    print(f"Best fitness (mAP@50-95): {best_params.get('fitness', 'N/A')}")
    print("\n📋 Copy these parameters for your final training:")
    print("-" * 70)
    
    # Print the key parameters
    key_params = ['lr0', 'lrf', 'momentum', 'weight_decay', 'box', 'cls', 'dfl', 
                  'hsv_h', 'hsv_s', 'hsv_v', 'degrees', 'translate', 'scale', 
                  'shear', 'perspective', 'flipud', 'fliplr', 'mosaic', 'mixup', 'copy_paste']
    
    for param in key_params:
        if param in best_params:
            print(f"    {param}={best_params[param]},")
    
    print("-" * 70)
    print("\n Use these in your final model.train() call!")
else:
    print(f"❌ YAML file not found at: {yaml_path}")

## Summary & Next Steps

- Area-ratio and aspect-ratio outlier thresholds were calibrated visually (Section 7.2), not guessed
- Duplicate/interchangeable boxes (same wheat head annotated twice) detected via pairwise IoU per image; one copy of each pair is now excluded from training labels (Section 9.1)
- Cleaned, attribute-separated box data saved to `outputs/train_boxes_clean.csv`, including `is_outlier`, `is_duplicate`, and `use_for_training` flags
- A source-stratified, image-level train/val split (85/15) was built and saved to `outputs/image_split.csv`, so the same image never appears on both sides
- An Albumentations augmentation pipeline (`train_transform`) was built and visually verified against real bounding boxes across images with different box densities — it stays a reusable pipeline object, applied on-the-fly to the train split only, not baked into static files
- YOLO-format dataset (`images/{train,val}` + `labels/{train,val}` + `data.yaml`) written to `outputs/yolo_dataset/`, excluding area/negative-dim outliers and dropped duplicate boxes

**Week 3 & 4** will pick up from `outputs/yolo_dataset/` and `train_transform` to
train a baseline detector (e.g. YOLOv5/v8 or Faster R-CNN) on the train split,
evaluate honestly on the untouched val split, and iterate on fine-tuning and
optimization (anchor tuning, threshold sweeps, TTA, architecture comparison, etc.).